# ⚽💰 Football Finance Intelligence
## Module 6 — Agents IA : un directeur sportif augmenté

**Idée :** donner à un modèle d'IA (Gemini) des **outils** (fonctions Python) pour interroger la base de données, utiliser le modèle de valorisation du Module 4 et rédiger des notes de recrutement.

🔑 **Principe clé : l'IA ne calcule jamais elle-même.** Les calculs sont réalisés par du code Python vérifiable. L'IA choisit les outils, lit leurs résultats et les interprète. Elle ne peut donc pas inventer de chiffres.

### Étapes
1. Connexion à Gemini
2. Création des outils
3. Un premier agent analyste
4. Un système multi-agents : un scout et un rédacteur

In [4]:
!pip install duckdb google-genai -q
!curl -sLO https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/transfermarkt-datasets.duckdb
!curl -sLO https://raw.githubusercontent.com/Ahmed-Badi3/football-finance-intelligence/main/valorisation_joueurs.csv

import duckdb
import pandas as pd
import numpy as np
from google import genai
from google.colab import userdata

# --- Les données ---
con = duckdb.connect("transfermarkt-datasets.duckdb", read_only=True)
def q(sql):
    return con.sql(sql).df()

valo = pd.read_csv("valorisation_joueurs.csv")
print(f"Fichier de valorisation du Module 4 : {len(valo):,} joueurs")

# --- Connexion à Gemini ---
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

print("\nModèles Flash disponibles :")
for m in client.models.list():
    if "flash" in m.name:
        print(" -", m.name)

# --- Premier test ---
MODELE = "gemini-3.8-flash"
reponse = client.models.generate_content(
    model=MODELE,
    contents="En une phrase, explique ce qu'est un agent IA à un étudiant en finance.",
)
print("\nRéponse de Gemini :")
print(reponse.text)

Fichier de valorisation du Module 4 : 7,287 joueurs

Modèles Flash disponibles :
 - models/gemini-2.5-flash
 - models/gemini-2.5-flash-preview-tts
 - models/gemini-flash-latest
 - models/gemini-flash-lite-latest
 - models/gemini-2.5-flash-lite
 - models/gemini-2.5-flash-image
 - models/gemini-3-flash-preview
 - models/gemini-3.1-flash-lite-preview
 - models/gemini-3.1-flash-lite
 - models/gemini-3.1-flash-image-preview
 - models/gemini-3.1-flash-image
 - models/gemini-3.1-flash-lite-image
 - models/gemini-3.5-flash
 - models/gemini-3.5-flash-lite
 - models/gemini-omni-flash-preview
 - models/gemini-omni-1.1-flash
 - models/gemini-3.6-flash
 - models/gemini-3.7-flash
 - models/gemini-3.8-flash
 - models/gemini-3.1-flash-tts-preview
 - models/gemini-3.8-flash-tts
 - models/gemini-3.8-flash-lite-tts
 - models/gemini-2.5-flash-native-audio-latest
 - models/gemini-2.5-flash-native-audio-preview-09-2025
 - models/gemini-2.5-flash-native-audio-preview-12-2025
 - models/gemini-3.1-flash-live-p

KeyboardInterrupt: 

In [5]:
import time
from google.genai import types

# Client avec une limite de 30 secondes par requête
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"),
                      http_options=types.HttpOptions(timeout=30000))

for nom in ["gemini-flash-lite-latest", "gemini-3.5-flash",
            "gemini-flash-latest", "gemini-3.8-flash"]:
    debut = time.time()
    try:
        r = client.models.generate_content(model=nom, contents="Réponds uniquement : OK")
        print(f"✅ {nom} : {r.text.strip()} ({time.time() - debut:.1f} s)")
    except Exception as e:
        print(f"❌ {nom} : {str(e)[:120]}")

✅ gemini-flash-lite-latest : OK (12.8 s)
✅ gemini-3.5-flash : OK (2.5 s)
❌ gemini-flash-latest : 504 DEADLINE_EXCEEDED. {'error': {'code': 504, 'message': 'Deadline expired before operation could complete.', 'status':
❌ gemini-3.8-flash : 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand 


## 1. Connexion à Gemini
Plusieurs modèles ont été testés avec une limite de 30 secondes par requête :
| Modèle | Résultat | Temps de réponse |
|---|---|---|
| gemini-3.5-flash | ✅ | 2,5 s |
| gemini-flash-lite-latest | ✅ | 12,8 s |
| gemini-flash-latest | ❌ délai dépassé | — |
| gemini-3.8-flash | ❌ forte demande | — |

**Modèle retenu : `gemini-3.5-flash`**, avec `gemini-flash-lite-latest` en solution de secours. Le nom du modèle est stocké dans une seule variable pour pouvoir le changer facilement.

## 2. Les outils de l'agent
| Outil | Rôle |
|---|---|
| `executer_sql` | Requête SQL en lecture seule sur la base Transfermarkt (50 lignes maximum) |
| `chercher_sous_evalues` | Recherche de joueurs sous-évalués à partir du modèle du Module 4 |
| `fiche_joueur` | Profil d'un joueur : club, contrat, estimation du modèle, historique de valeur |

Chaque outil est **testé indépendamment** avant d'être confié à l'agent.

In [7]:
MODELE = "gemini-3.5-flash"
MODELE_SECOURS = "gemini-flash-lite-latest"

def executer_sql(requete: str) -> str:
    """Exécute une requête SQL en lecture seule (SELECT) sur la base Transfermarkt
    et renvoie au maximum 50 lignes de résultat sous forme de texte."""
    texte = requete.strip().rstrip(";")
    if not texte.lower().startswith(("select", "with")):
        return "Refusé : seules les requêtes SELECT sont autorisées."
    try:
        resultat = q(f"SELECT * FROM ({texte}) LIMIT 50")
        return resultat.to_string(index=False) if len(resultat) else "Aucun résultat."
    except Exception as e:
        return f"Erreur SQL : {e}"


def chercher_sous_evalues(poste: str = "", age_max: float = 40,
                          valeur_max_M: float = 1000, ligue: str = "",
                          nationalite: str = "", nb_resultats: int = 10) -> str:
    """Cherche les joueurs les plus sous-évalués selon le modèle de valorisation.
    poste : 'Defender', 'Midfield', 'Attack' ou 'Goalkeeper' (vide = tous).
    ligue : code du championnat, par exemple 'GB1' (Angleterre), 'ES1' (Espagne),
            'IT1' (Italie), 'L1' (Allemagne), 'FR1' (France), 'PO1' (Portugal),
            'NL1' (Pays-Bas), 'BE1' (Belgique) (vide = tous).
    nationalite : en anglais, par exemple 'Morocco' (vide = toutes).
    Seuls les joueurs ayant joué au moins 900 minutes cette saison sont retenus.
    Le ratio = valeur estimée / valeur réelle : un ratio de 2 signifie que le
    modèle estime le joueur à deux fois sa valeur actuelle."""
    df = valo[(valo["minutes_n"] >= 900) & (valo["age"] <= age_max)
              & (valo["valeur_M"] <= valeur_max_M)]
    if poste:
        df = df[df["position"] == poste]
    if ligue:
        df = df[df["ligue"] == ligue]
    if nationalite:
        df = df[df["nationalite"] == nationalite]
    if df.empty:
        return "Aucun joueur ne correspond à ces critères."
    colonnes = ["name", "club", "ligue", "nationalite", "position", "age",
                "minutes_n", "valeur_M", "valeur_estimee_M", "ratio"]
    return df.nlargest(int(nb_resultats), "ratio")[colonnes].round(2).to_string(index=False)


def fiche_joueur(nom: str) -> str:
    """Renvoie le profil d'un joueur : club, championnat, âge, poste, contrat,
    statistiques de la saison, valeur réelle, valeur estimée par le modèle
    et historique de sa valeur sur les dernières années."""
    trouves = valo[valo["name"].str.contains(nom, case=False, na=False)]
    if trouves.empty:
        return f"Aucun joueur trouvé pour '{nom}' dans le fichier de valorisation."
    j = trouves.iloc[0]
    histo = q(f"""
        SELECT YEAR(date) AS annee,
               ROUND(arg_max(market_value_in_eur, date) / 1e6, 1) AS valeur_M
        FROM player_valuations
        WHERE player_id = {int(j['player_id'])}
        GROUP BY 1 ORDER BY 1 DESC LIMIT 6
    """)
    return (f"{j['name']} | {j['club']} ({j['ligue']}) | {j['nationalite']} | "
            f"{j['position']} ({j['sub_position']}) | {j['age']:.1f} ans\n"
            f"Contrat restant : {j['annees_contrat']:.1f} ans | Sélections : {j['selections']}\n"
            f"Saison : {j['minutes_n']:.0f} minutes, {j['buts_n']:.0f} buts, "
            f"{j['passes_n']:.0f} passes décisives\n"
            f"Valeur réelle : {j['valeur_M']:.1f} M€ | Valeur estimée : {j['valeur_estimee_M']:.1f} M€ "
            f"| Ratio : {j['ratio']:.2f}\n"
            f"Historique de valeur (M€) :\n{histo.to_string(index=False)}")


# --- Tests des outils, sans IA ---
print("=== Test 1 : executer_sql ===")
print(executer_sql("SELECT name, market_value_in_eur FROM players "
                   "WHERE country_of_citizenship = 'Morocco' "
                   "ORDER BY market_value_in_eur DESC NULLS LAST LIMIT 3"))
print("\n=== Test 2 : sécurité ===")
print(executer_sql("DROP TABLE players"))
print("\n=== Test 3 : chercher_sous_evalues ===")
print(chercher_sous_evalues(poste="Defender", age_max=23, ligue="FR1", nb_resultats=5))
print("\n=== Test 4 : fiche_joueur ===")
print(fiche_joueur("Hakimi"))

=== Test 1 : executer_sql ===
           name  market_value_in_eur
  Achraf Hakimi             80000000
 Ayyoub Bouaddi             50000000
Abde Ezzalzouli             40000000

=== Test 2 : sécurité ===
Refusé : seules les requêtes SELECT sont autorisées.

=== Test 3 : chercher_sous_evalues ===
              name             club ligue  nationalite position   age  minutes_n  valeur_M  valeur_estimee_M  ratio
          Juma Bah         OGC Nice   FR1 Sierra Leone Defender 20.22     2660.0      10.0             24.80   2.48
Kojo Peprah Oppong         OGC Nice   FR1        Ghana Defender 22.07     2973.0      10.0             16.05   1.60
  Mahamadou Nagida Stade Rennais FC   FR1     Cameroon Defender 21.01     1036.0       7.0             10.05   1.44
     Antoine Mendy         OGC Nice   FR1      Senegal Defender 22.09     2768.0      10.0             13.76   1.38
      Sadibou Sané          FC Metz   FR1      Senegal Defender 22.06     1941.0       7.0              9.11   1.30

=== T

### Résultats des tests
| Test | Résultat |
|---|---|
| Requête SQL | ✅ Joueurs marocains les plus chers : Hakimi (80 M€), Bouaddi (50 M€), Ezzalzouli (40 M€) |
| Sécurité | ✅ Requête de suppression (`DROP TABLE`) refusée |
| Recherche de sous-évalués | ✅ Jeunes défenseurs de Ligue 1 : Juma Bah (Nice) en tête, ratio 2,48 |
| Fiche joueur | ✅ Profil complet d'Achraf Hakimi, dont l'historique de valeur (70 → 60 → 80 M€) |

## 3. Le premier agent : un analyste financier du football
- **Instructions système :** le rôle de l'agent, ses règles (ne jamais inventer de chiffres, toujours utiliser les outils, signaler les limites) et la description de la base de données.
- **Température 0,2 :** réponses rigoureuses et reproductibles, adaptées à l'analyse financière.
- **Appel automatique des outils :** Gemini choisit lui-même l'outil et ses paramètres. Chaque appel est affiché pour rendre le raisonnement transparent.
- **Résilience :** bascule automatique vers le modèle de secours en cas d'indisponibilité.

In [8]:
from IPython.display import Markdown, display

# Client avec une limite de 60 secondes (les agents font parfois plusieurs appels)
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"),
                      http_options=types.HttpOptions(timeout=60000))

INSTRUCTIONS = """
Tu es un analyste financier spécialisé dans le football, au service d'un directeur sportif.

RÈGLES ABSOLUES
1. N'invente JAMAIS de chiffre. Chaque chiffre doit venir d'un outil.
2. Utilise toujours les outils pour répondre. Si aucun outil ne permet de répondre, dis-le clairement.
3. Réponds en français, de façon structurée : réponse directe, chiffres clés, puis limites.
4. Signale les limites : joueurs prêtés (club propriétaire différent du club réel),
   championnats non couverts, estimations du modèle qui ne mesurent ni le potentiel ni les blessures.

OUTILS
- chercher_sous_evalues : recherche de joueurs sous-évalués (modèle de machine learning).
- fiche_joueur : profil complet d'un joueur.
- executer_sql : toute autre question sur la base (transferts, clubs, valeurs, matchs).

BASE DE DONNÉES (DuckDB) — montants en euros
- players(player_id, name, position, sub_position, date_of_birth, country_of_citizenship,
  current_club_name, current_club_domestic_competition_id, market_value_in_eur,
  contract_expiration_date, international_caps)
- transfers(player_id, player_name, transfer_date, transfer_season, from_club_name,
  to_club_name, transfer_fee, market_value_in_eur) — transfer_season au format '25/26'
- clubs(club_id, name, domestic_competition_id, total_market_value)
- player_valuations(player_id, date, market_value_in_eur)
- appearances(player_id, game_id, date, competition_id, goals, assists, minutes_played)
- games(game_id, competition_id, season, date, home_club_id, away_club_id,
  home_club_goals, away_club_goals)
Codes des championnats : GB1 Angleterre, ES1 Espagne, IT1 Italie, L1 Allemagne,
FR1 France, PO1 Portugal, NL1 Pays-Bas, BE1 Belgique.
"""

OUTILS = [executer_sql, chercher_sous_evalues, fiche_joueur]

def demander(question, modele=MODELE):
    config = types.GenerateContentConfig(system_instruction=INSTRUCTIONS,
                                         tools=OUTILS, temperature=0.2)
    try:
        r = client.models.generate_content(model=modele, contents=question, config=config)
    except Exception as e:
        if modele != MODELE_SECOURS:
            print(f"⚠️ {modele} indisponible, passage au modèle de secours...")
            return demander(question, MODELE_SECOURS)
        return f"Erreur : {e}"
    # Afficher les outils appelés par l'agent
    for etape in (r.automatic_function_calling_history or []):
        for part in (etape.parts or []):
            if part.function_call:
                print(f"🔧 L'agent appelle : {part.function_call.name}"
                      f"({dict(part.function_call.args or {})})")
    return r.text

print("✅ Agent prêt")

✅ Agent prêt


In [9]:
display(Markdown(demander(
    "Quels sont les 5 transferts les plus chers de l'histoire selon la base ?"
)))

🔧 L'agent appelle : executer_sql({'requete': 'SELECT player_name, transfer_date, transfer_season, from_club_name, to_club_name, transfer_fee, market_value_in_eur \nFROM transfers \nWHERE transfer_fee IS NOT NULL \nORDER BY transfer_fee DESC \nLIMIT 10'})


Voici les 5 transferts les plus chers de l'histoire enregistrés dans la base de données.

---

### Réponse directe
Le transfert le plus cher de l'histoire dans la base de données est celui de **Neymar** du FC Barcelone au Paris Saint-Germain pour **222 millions d'euros**, suivi de Kylian Mbappé, Ousmane Dembélé, Alexander Isak et Philippe Coutinho.

---

### Chiffres clés (Top 5)

1. **Neymar** — **222 000 000 €**
   - **Provenance** : FC Barcelone ➡️ **Destination** : Paris Saint-Germain
   - **Date / Saison** : 03/08/2017 (Saison 17/18)
   - **Valeur marchande à la date du transfert** : 100 000 000 €

2. **Kylian Mbappé** — **180 000 000 €**
   - **Provenance** : AS Monaco ➡️ **Destination** : Paris Saint-Germain
   - **Date / Saison** : 01/07/2018 (Saison 18/19)
   - **Valeur marchande à la date du transfert** : 120 000 000 €

3. **Ousmane Dembélé** — **148 000 000 €**
   - **Provenance** : Borussia Dortmund ➡️ **Destination** : FC Barcelone
   - **Date / Saison** : 25/08/2017 (Saison 17/18)
   - **Valeur marchande à la date du transfert** : 33 000 000 €

4. **Alexander Isak** — **145 000 000 €**
   - **Provenance** : Newcastle United ➡️ **Destination** : Liverpool FC
   - **Date / Saison** : 01/09/2025 (Saison 25/26)
   - **Valeur marchande à la date du transfert** : 120 000 000 €

5. **Philippe Coutinho** — **135 000 000 €**
   - **Provenance** : Liverpool FC ➡️ **Destination** : FC Barcelone
   - **Date / Saison** : 08/01/2018 (Saison 17/18)
   - **Valeur marchande à la date du transfert** : 90 000 000 €

---

### Limites & Remarques
* **Données prospectives ou prévisionnelles** : La base contient des transactions datées dans le futur (ex. Alexander Isak daté de septembre 2025).
* **Joueurs en prêt avec option d'achat** : La date de la transaction enregistrée peut correspondre à la levée d'option définitive plutôt qu'au début du prêt initial (ex. Mbappé arrivé au PSG en prêt à l'été 2017, transfert comptabilisé au 01/07/2018).
* **Détail des montants** : Les chiffres représentent l'indemnité globale répertoriée par Transfermarkt, sans ventilation entre montant fixe, bonus et commissions d'agents.
* **Inflation et couverture** : Les montants ne sont pas ajustés de l'inflation économique ou de l'inflation du marché des transferts, et seuls les championnats et périodes couverts par Transfermarkt sont pris en compte.

In [10]:
display(Markdown(demander(
    "Je cherche un milieu de terrain de 23 ans maximum, sous-évalué, qui joue "
    "au Portugal ou aux Pays-Bas, pour moins de 15 M€. Propose-moi 3 profils "
    "et explique ton choix."
)))

🔧 L'agent appelle : chercher_sous_evalues({'valeur_max_M': 15, 'poste': 'Midfield', 'nb_resultats': 10, 'age_max': 23, 'ligue': 'PO1'})
🔧 L'agent appelle : chercher_sous_evalues({'valeur_max_M': 15, 'ligue': 'NL1', 'nb_resultats': 10, 'poste': 'Midfield', 'age_max': 23})
🔧 L'agent appelle : fiche_joueur({'nom': 'Dave Kwakman'})
🔧 L'agent appelle : fiche_joueur({'nom': 'Jorthy Mokio'})
🔧 L'agent appelle : fiche_joueur({'nom': 'Diego Rodrigues'})
🔧 L'agent appelle : fiche_joueur({'nom': 'Kristian Hlynsson'})
🔧 L'agent appelle : executer_sql({'requete': "SELECT p.player_id, p.name, p.current_club_name, t.from_club_name, t.to_club_name, t.transfer_fee\nFROM players p\nLEFT JOIN transfers t ON p.player_id = t.player_id\nWHERE p.name IN ('Dave Kwakman', 'Jorthy Mokio', 'Diego Rodrigues', 'Kristian Hlynsson', 'Espen van Ee')\n"})
🔧 L'agent appelle : fiche_joueur({'nom': 'Espen van Ee'})
🔧 L'agent appelle : executer_sql({'requete': "SELECT player_id, name, current_club_name, contract_expiratio

Voici 3 profils de milieux de terrain de 23 ans maximum évoluant aux Pays-Bas (Eredivisie) ou au Portugal (Liga Portugal), d'une valeur marchande inférieure à 15 M€ et considérés comme sous-évalués par le modèle de valorisation.

---

### Chiffres clés

| Joueur | Club (Ligue) | Âge | Poste principal | Valeur réelle | Valeur estimée | Ratio (Estimée / Réelle) | Temps de jeu & Stats (saison) | Contrat restant |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Dave Kwakman** | AZ Alkmaar (NL1) | 21,9 ans | Milieu central | **1,5 M€** | **3,2 M€** | **2,10** | 1 034 min (0 B, 4 PD) | 3,0 ans |
| **Jorthy Mokio** | Ajax Amsterdam (NL1) | 18,3 ans | Milieu défensif | **8,0 M€** | **13,6 M€** | **1,70** | 1 836 min (4 B, 3 PD) | 5,0 ans |
| **Diego Rodrigues** | SC Braga (PO1) | 21,1 ans | Milieu central | **5,0 M€** | **7,3 M€** | **1,46** | 1 208 min (2 B, 4 PD) | 4,0 ans |

*(Alternative à profil plus offensif : **Kristian Hlynsson**, FC Twente — 22,4 ans, MOC, valeur réelle 6,0 M€, estimée 8,8 M€, ratio 1,46, 2 852 min, 12 buts, 4 passes décisives).*

---

### Explication des choix

1. **Dave Kwakman (AZ Alkmaar — Eredivisie)**  
   * **Opportunité financière :** Il présente le ratio de sous-évaluation le plus élevé (2,10) parmi les milieux retenus. 
   * **Profil :** Milieu central créatif, capable de délivrer 4 passes décisives en un peu plus de 1 000 minutes de jeu. Avec une valeur marchande actuelle de seulement 1,5 M€, il constitue un pari très accessible et à faible risque financier.

2. **Jorthy Mokio (Ajax Amsterdam — Eredivisie)**  
   * **Profil haut niveau :** À seulement 18,3 ans, il totalise déjà 1 836 minutes de jeu avec l'Ajax et compte 6 sélections internationales.
   * **Polyvalence & Impact :** Milieu défensif capable de peser offensivement (4 buts, 3 passes décisives). Le modèle l'estime à 13,6 M€ contre une valeur de marché de 8,0 M€ (ratio 1,70), ce qui reflète son temps de jeu important pour son jeune âge.

3. **Diego Rodrigues (SC Braga — Liga Portugal)**  
   * **Meilleure opportunité au Portugal :** Il s'agit du milieu le plus sous-évalué du championnat portugais sous la barre des 15 M€ (ratio 1,46).
   * **Équilibre :** Milieu central complet (2 buts, 4 passes décisives en 1 208 min) disposant de 4 ans de contrat, estimé à 7,3 M€ pour une valeur de marché de 5,0 M€.

---

### Limites de l'analyse

* **Limites du modèle de valorisation :** L'estimation du modèle se base sur le temps de jeu effectif et les métriques de la saison en cours. Elle ne prend pas en compte le potentiel intrinsèque de développement à long terme, la santé médicale/l'historique des blessures, ni les clauses libératoires éventuelles.
* **Situation contractuelle :** Aucun de ces trois joueurs n'est actuellement prêté (ils appartiennent tous à leur club actuel avec des contrats de 3 à 5 ans), ce qui implique une négociation de transfert ferme avec leur club propriétaire.
* **Périmètre des championnats :** Seuls les championnats couverts par la base (GB1, ES1, IT1, L1, FR1, PO1, NL1, BE1) sont pris en compte dans cette recherche.

In [11]:
display(Markdown(demander(
    "Quelle sera la valeur de Bilal El Khannouss en 2030 ?"
)))

🔧 L'agent appelle : fiche_joueur({'nom': 'Bilal El Khannouss'})


### Réponse directe
**Il est impossible de prédire la valeur marchande de Bilal El Khannouss en 2030.** La base de données et le modèle de valorisation disponibles ne permettent pas de réaliser des projections financières à si long terme. Le modèle évalue la valeur actuelle d'un joueur en fonction de ses performances récentes, mais ne prédit pas l'évolution future des valeurs sur plusieurs années.

---

### Chiffres clés (situation actuelle)
* **Joueur :** Bilal El Khannouss
* **Club actuel :** VfB Stuttgart (Allemagne - L1)
* **Âge :** 22,1 ans
* **Nationalité :** Maroc (43 sélections)
* **Poste :** Milieu offensif
* **Durée de contrat restante :** 4,0 ans
* **Statistiques cette saison :** 3 182 minutes jouées, 9 buts, 7 passes décisives
* **Valeur marchande réelle actuelle (2026) :** 35,0 M€
* **Valeur estimée par le modèle :** 42,1 M€ (Ratio : 1,20 — joueur légèrement sous-évalué)

**Évolution historique de sa valeur marchande :**
* 2022 : 4,5 M€
* 2023 : 22,0 M€
* 2024 : 30,0 M€
* 2025 : 32,0 M€
* 2026 : 35,0 M€

---

### Limites
* **Pas de modèle prédictif à futur :** Le modèle de valorisation mesure la valeur marchande d'un joueur à l'instant T sur la base de son temps de jeu et de ses statistiques récentes. Il ne s'agit pas d'un outil de projection du potentiel à horizon 2030.
* **Aléas de carrière non modélisables :** La valeur en 2030 dépendra de facteurs imprévisibles tels que les blessures, l'évolution de ses performances, la durée de son contrat à cette échéance, l'inflation du marché des transferts ou d'éventuels transferts futurs.

### Lecture des tests de l'agent
| Question | Outils appelés | Résultat |
|---|---|---|
| 5 transferts les plus chers | 1 requête SQL écrite par l'agent | ✅ Neymar, Mbappé, Dembélé, Isak, Coutinho |
| Milieux sous-évalués (Portugal, Pays-Bas, < 15 M€) | 9 appels : 2 recherches, 5 fiches, 2 requêtes SQL de vérification | ✅ Kwakman (AZ), Mokio (Ajax), Rodrigues (Braga) |
| Piège : valeur d'El Khannouss en 2030 | 1 fiche joueur | ✅ Refus d'inventer un chiffre, analyse de la situation actuelle |

- **Planification autonome :** pour la 2ᵉ question, l'agent a décomposé la recherche, approfondi chaque candidat et vérifié de lui-même les transferts (prêts) et les contrats.
- **Honnêteté :** les règles du système empêchent l'invention de chiffres.

⚠️ **Défaut identifié :** l'agent a qualifié de « futur » un transfert de septembre 2025, car le modèle ne connaît pas la date du jour. **Correction :** la date du jour est ajoutée aux instructions.

In [12]:
from datetime import date
INSTRUCTIONS = f"Date du jour : {date.today():%d/%m/%Y}. Toute date antérieure est passée.\n" + INSTRUCTIONS

## 4. Le comité de recrutement multi-agents
Trois agents spécialisés travaillent en pipeline, chacun recevant le travail du précédent :
| Agent | Rôle | Outils |
|---|---|---|
| 🕵️ Scout | Identifier 3 joueurs correspondant au besoin | Recherche de sous-évalués, fiches, SQL |
| 💰 Analyste financier | Évaluer coût, amortissement et risques | Fiches, SQL, simulation d'amortissement |
| ✍️ Rédacteur | Rédiger la note de recrutement | Aucun (rédaction uniquement) |

**Nouvel outil — `simuler_amortissement` :** charge annuelle, valeur nette comptable et plus-value latente (IAS 38), dans la continuité du Module 2.

La note finale est enregistrée dans le fichier `note_recrutement.md`.

In [13]:
def simuler_amortissement(prix_achat_M: float, duree_contrat_ans: int,
                          valeur_estimee_M: float) -> str:
    """Simule l'amortissement linéaire d'un transfert (norme IAS 38).
    Pour chaque année du contrat : charge d'amortissement, valeur nette comptable (VNC)
    et plus-value latente si le joueur était revendu à sa valeur estimée par le modèle."""
    duree = max(int(duree_contrat_ans), 1)
    charge = prix_achat_M / duree
    lignes = [f"Prix d'achat : {prix_achat_M:.1f} M€ | Contrat : {duree} ans | "
              f"Charge annuelle : {charge:.2f} M€",
              "Année | VNC fin d'année (M€) | Plus-value latente si revente à la valeur estimée (M€)"]
    for annee in range(1, duree + 1):
        vnc = prix_achat_M - charge * annee
        lignes.append(f"{annee} | {vnc:.1f} | {valeur_estimee_M - vnc:+.1f}")
    return "\n".join(lignes)


def agent(nom, role, consigne, outils, modele=MODELE):
    """Un agent générique : un rôle, une consigne, une liste d'outils."""
    reglages = dict(system_instruction=role, temperature=0.2)
    if outils:
        reglages["tools"] = outils
    try:
        r = client.models.generate_content(model=modele, contents=consigne,
                                           config=types.GenerateContentConfig(**reglages))
    except Exception as e:
        if modele != MODELE_SECOURS:
            print(f"   ⚠️ {modele} indisponible, passage au modèle de secours...")
            return agent(nom, role, consigne, outils, MODELE_SECOURS)
        return f"Erreur : {e}"
    for etape in (r.automatic_function_calling_history or []):
        for part in (etape.parts or []):
            if part.function_call:
                print(f"   🔧 {nom} appelle : {part.function_call.name}")
    return r.text


ROLE_SCOUT = INSTRUCTIONS + """
TON RÔLE : SCOUT.
Propose 3 joueurs maximum correspondant au besoin du club.
Pour chacun : club, championnat, âge, poste, minutes jouées, valeur réelle, valeur estimée,
ratio et contrat restant, uniquement à partir des outils.
Vérifie avec executer_sql si un joueur est prêté (dernier transfert vers son club actuel).
"""

ROLE_ANALYSTE = INSTRUCTIONS + """
TON RÔLE : DIRECTEUR FINANCIER.
Pour chaque joueur proposé par le scout :
1. Hypothèse de prix d'achat = valeur réelle actuelle. Contrat proposé : 5 ans.
2. Utilise simuler_amortissement (prix, 5 ans, valeur estimée).
3. Évalue les risques : âge (la valeur progresse jusqu'à environ 23 ans et se déprécie
   fortement après 29 ans), contrat restant (moins de 2 ans = levier de négociation),
   ratio du modèle, temps de jeu.
4. Classe les joueurs du plus intéressant au moins intéressant, en justifiant.
"""

ROLE_REDACTEUR = """
Tu es le rédacteur du comité de recrutement d'un club de football.
Rédige en français une NOTE DE RECRUTEMENT professionnelle, en Markdown, avec :
1. Un titre et la date
2. Un résumé exécutif (3 lignes maximum) avec la recommandation finale
3. Un tableau comparatif des joueurs
4. Une analyse par joueur (atouts, risques, impact financier)
5. Les risques et limites de l'analyse
6. Les prochaines étapes (analyse vidéo, visite médicale, négociation)
RÈGLE ABSOLUE : n'ajoute AUCUN chiffre absent des documents fournis.
"""

print("✅ Outil d'amortissement et rôles prêts")
print(simuler_amortissement(10, 5, 24.8))   # test sans IA

✅ Outil d'amortissement et rôles prêts
Prix d'achat : 10.0 M€ | Contrat : 5 ans | Charge annuelle : 2.00 M€
Année | VNC fin d'année (M€) | Plus-value latente si revente à la valeur estimée (M€)
1 | 8.0 | +16.8
2 | 6.0 | +18.8
3 | 4.0 | +20.8
4 | 2.0 | +22.8
5 | 0.0 | +24.8


### Lecture : premier test du comité
**Ce qui fonctionne :** les trois agents travaillent en chaîne, la note est structurée, et la bascule automatique vers le modèle de secours a fonctionné à deux reprises.

**⚠️ Faux positif :** le comité recommande en priorité Jayden Meghoma (ratio 14,37). Or ce joueur, enregistré à Brentford, a joué toute la saison **prêté aux Rangers** : c'est l'erreur identifiée au Module 4. La note affirme même qu'aucun joueur n'est prêté, car le scout a cherché le prêt dans la table des transferts, où il n'apparaît pas.

💡 **Leçon : les agents amplifient les erreurs des données.** Le biais des prêts avait été identifié au Module 4 mais pas corrigé dans les données. L'agent l'a repris et justifié de façon convaincante. **La solution consiste à corriger les données à la source, pas à reformuler la consigne.**

**Corrections apportées :**
1. Calcul du **club réel** de chaque joueur (club où il a joué le plus de minutes en 2025/26) et signalement automatique des joueurs dont le club réel diffère du club enregistré (prêt ou transfert en cours de saison).
2. Exclusion par défaut de ces joueurs dans la recherche de sous-évalués.
3. Ajout de la date du jour pour le rédacteur.

In [15]:
# --- 1. Le club réel de chaque joueur en 2025/26 ---
clubs_reels = q("""
WITH minutes_par_club AS (
    SELECT a.player_id, a.player_club_id AS club_reel_id, c.name AS club_reel,
           SUM(a.minutes_played) AS minutes,
           ROW_NUMBER() OVER (PARTITION BY a.player_id
                              ORDER BY SUM(a.minutes_played) DESC) AS rang
    FROM appearances a
    JOIN games g ON a.game_id = g.game_id
    JOIN clubs c ON a.player_club_id = c.club_id
    WHERE TRY_CAST(g.season AS INT) = 2025
    GROUP BY a.player_id, a.player_club_id, c.name
)
SELECT m.player_id, m.club_reel,
       TRY_CAST(p.current_club_id AS INT) <> m.club_reel_id AS pret_probable
FROM minutes_par_club m
JOIN players p ON m.player_id = p.player_id
WHERE m.rang = 1
""")

valo = valo.drop(columns=["club_reel", "pret_probable"], errors="ignore")
valo = valo.merge(clubs_reels, on="player_id", how="left")
valo["pret_probable"] = valo["pret_probable"].fillna(False).astype(bool)
print(f"Joueurs dont le club réel diffère du club enregistré : {valo['pret_probable'].sum():,}")


# --- 2. Les outils corrigés ---
def chercher_sous_evalues(poste: str = "", age_max: float = 40,
                          valeur_max_M: float = 1000, ligue: str = "",
                          nationalite: str = "", nb_resultats: int = 10,
                          inclure_prets: bool = False) -> str:
    """Cherche les joueurs les plus sous-évalués selon le modèle de valorisation.
    poste : 'Defender', 'Midfield', 'Attack' ou 'Goalkeeper' (vide = tous).
    ligue : code du championnat, par exemple 'GB1', 'ES1', 'IT1', 'L1', 'FR1',
            'PO1', 'NL1', 'BE1' (vide = tous).
    nationalite : en anglais, par exemple 'Morocco' (vide = toutes).
    inclure_prets : False par défaut. Les joueurs qui jouent dans un autre club que
    leur club enregistré (prêt probable) sont alors exclus, car leur estimation est biaisée.
    Seuls les joueurs ayant joué au moins 900 minutes cette saison sont retenus.
    Ratio = valeur estimée / valeur réelle."""
    df = valo[(valo["minutes_n"] >= 900) & (valo["age"] <= age_max)
              & (valo["valeur_M"] <= valeur_max_M)]
    if not inclure_prets:
        df = df[~df["pret_probable"]]
    if poste:
        df = df[df["position"] == poste]
    if ligue:
        df = df[df["ligue"] == ligue]
    if nationalite:
        df = df[df["nationalite"] == nationalite]
    if df.empty:
        return "Aucun joueur ne correspond à ces critères."
    colonnes = ["name", "club", "club_reel", "ligue", "nationalite", "position", "age",
                "minutes_n", "valeur_M", "valeur_estimee_M", "ratio"]
    return df.nlargest(int(nb_resultats), "ratio")[colonnes].round(2).to_string(index=False)


def fiche_joueur(nom: str) -> str:
    """Renvoie le profil d'un joueur : club enregistré, club réel de la saison,
    championnat, âge, poste, contrat, statistiques, valeur réelle, valeur estimée
    par le modèle et historique de sa valeur. Signale les prêts probables."""
    trouves = valo[valo["name"].str.contains(nom, case=False, na=False)]
    if trouves.empty:
        return f"Aucun joueur trouvé pour '{nom}' dans le fichier de valorisation."
    j = trouves.iloc[0]
    histo = q(f"""
        SELECT YEAR(date) AS annee,
               ROUND(arg_max(market_value_in_eur, date) / 1e6, 1) AS valeur_M
        FROM player_valuations
        WHERE player_id = {int(j['player_id'])}
        GROUP BY 1 ORDER BY 1 DESC LIMIT 6
    """)
    alerte = ""
    if j["pret_probable"]:
        alerte = (f"\n⚠️ ALERTE : le joueur est enregistré à {j['club']} mais a joué la saison "
                  f"à {j['club_reel']} (prêt probable). Son estimation est BIAISÉE à la hausse "
                  f"et ne doit pas servir de base à une recommandation.")
    return (f"{j['name']} | Club enregistré : {j['club']} ({j['ligue']}) | "
            f"Club réel cette saison : {j['club_reel']} | {j['nationalite']} | "
            f"{j['position']} ({j['sub_position']}) | {j['age']:.1f} ans\n"
            f"Contrat restant : {j['annees_contrat']:.1f} ans | Sélections : {j['selections']}\n"
            f"Saison : {j['minutes_n']:.0f} minutes, {j['buts_n']:.0f} buts, "
            f"{j['passes_n']:.0f} passes décisives\n"
            f"Valeur réelle : {j['valeur_M']:.1f} M€ | Valeur estimée : {j['valeur_estimee_M']:.1f} M€ "
            f"| Ratio : {j['ratio']:.2f}{alerte}\n"
            f"Historique de valeur (M€) :\n{histo.to_string(index=False)}")


# --- 3. La date pour le rédacteur ---
if "Date du jour" not in ROLE_REDACTEUR:
    ROLE_REDACTEUR = f"Date du jour : {date.today():%d/%m/%Y}.\n" + ROLE_REDACTEUR

# --- 4. Vérifications ---
print("\n=== Fiche de Meghoma ===")
print(fiche_joueur("Meghoma"))
print("\n=== Jeunes défenseurs sous-évalués (prêts exclus) ===")
print(chercher_sous_evalues(poste="Defender", age_max=23, valeur_max_M=15, nb_resultats=5))

Joueurs dont le club réel diffère du club enregistré : 359

=== Fiche de Meghoma ===
Jayden Meghoma | Club enregistré : Brentford FC (GB1) | Club réel cette saison : Rangers FC | England | Defender (Left-Back) | 20.0 ans
Contrat restant : 2.0 ans | Sélections : 3
Saison : 2798 minutes, 1 buts, 6 passes décisives
Valeur réelle : 2.0 M€ | Valeur estimée : 28.7 M€ | Ratio : 14.37
⚠️ ALERTE : le joueur est enregistré à Brentford FC mais a joué la saison à Rangers FC (prêt probable). Son estimation est BIAISÉE à la hausse et ne doit pas servir de base à une recommandation.
Historique de valeur (M€) :
 annee  valeur_M
  2026       2.0
  2025       0.8

=== Jeunes défenseurs sous-évalués (prêts exclus) ===
               name            club       club_reel ligue   nationalite position   age  minutes_n  valeur_M  valeur_estimee_M  ratio
  Miguel Freckleton   St. Mirren FC   St. Mirren FC   SC1       England Defender 22.90     3330.0       0.2              0.66   3.29
Ethan Schilte-Brown   Kil

/tmp/ipykernel_2939/3702714868.py:23: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  valo["pret_probable"] = valo["pret_probable"].fillna(False).astype(bool)


### Vérification de la correction
- **359 joueurs** (environ 5 %) évoluent dans un autre club que leur club enregistré.
- La fiche de Jayden Meghoma affiche désormais une alerte (club réel : Rangers FC), et le joueur est exclu de la recherche.

### ⚠️ Nouveau biais : les micro-valeurs
La recherche fait remonter des joueurs valant 0,1 à 0,3 M€, dont le ratio est élevé pour un écart absolu minime (exemple : 0,1 → 0,28 M€, ratio 2,84). Or le modèle se trompe de 43 % en médiane sous 0,5 M€ (Module 4) : ces écarts relèvent surtout du **bruit statistique**.
**Correction :** ajout d'une valeur minimale (1 M€ par défaut), seuil à partir duquel le modèle est plus fiable.

In [16]:
def chercher_sous_evalues(poste: str = "", age_max: float = 40,
                          valeur_min_M: float = 1.0, valeur_max_M: float = 1000,
                          ligue: str = "", nationalite: str = "",
                          nb_resultats: int = 10, inclure_prets: bool = False) -> str:
    """Cherche les joueurs les plus sous-évalués selon le modèle de valorisation.
    poste : 'Defender', 'Midfield', 'Attack' ou 'Goalkeeper' (vide = tous).
    valeur_min_M : 1 M€ par défaut. En dessous, le modèle est peu fiable (erreur > 40 %).
    ligue : code du championnat, par exemple 'GB1', 'ES1', 'IT1', 'L1', 'FR1',
            'PO1', 'NL1', 'BE1' (vide = tous).
    nationalite : en anglais, par exemple 'Morocco' (vide = toutes).
    inclure_prets : False par défaut, pour exclure les prêts probables (estimation biaisée).
    Seuls les joueurs ayant joué au moins 900 minutes cette saison sont retenus.
    Ratio = valeur estimée / valeur réelle."""
    df = valo[(valo["minutes_n"] >= 900) & (valo["age"] <= age_max)
              & (valo["valeur_M"] >= valeur_min_M) & (valo["valeur_M"] <= valeur_max_M)]
    if not inclure_prets:
        df = df[~df["pret_probable"]]
    if poste:
        df = df[df["position"] == poste]
    if ligue:
        df = df[df["ligue"] == ligue]
    if nationalite:
        df = df[df["nationalite"] == nationalite]
    if df.empty:
        return "Aucun joueur ne correspond à ces critères."
    colonnes = ["name", "club", "club_reel", "ligue", "nationalite", "position", "age",
                "minutes_n", "valeur_M", "valeur_estimee_M", "ratio"]
    return df.nlargest(int(nb_resultats), "ratio")[colonnes].round(2).to_string(index=False)


print(chercher_sous_evalues(poste="Defender", age_max=23, valeur_max_M=15, nb_resultats=5))

               name                club           club_reel ligue   nationalite position   age  minutes_n  valeur_M  valeur_estimee_M  ratio
           Juma Bah            OGC Nice            OGC Nice   FR1  Sierra Leone Defender 20.22     2660.0      10.0             24.80   2.48
     Emmanuel Kakou       Cercle Brugge       Cercle Brugge   BE1 Cote d'Ivoire Defender 21.00     1733.0       2.0              4.49   2.24
      Colby Donovan           Celtic FC           Celtic FC   SC1      Scotland Defender 19.81     1287.0       1.5              2.80   1.86
Andreas Verstraeten    Royal Antwerp FC    Royal Antwerp FC   BE1       Belgium Defender 20.47      921.0       1.0              1.85   1.85
      Aurèle Amenda Eintracht Frankfurt Eintracht Frankfurt    L1   Switzerland Defender 22.92     1899.0      10.0             17.94   1.79


### Bilan des corrections de l'outil de recherche
| Version | N° 1 de la liste | Problème |
|---|---|---|
| Initiale | Jayden Meghoma (ratio 14,4) | Prêt aux Rangers non détecté |
| Après correction des prêts | Miguel Freckleton (0,2 M€) | Micro-valeur : bruit statistique |
| **Finale** | **Juma Bah** (Nice, 10 M€, ratio 2,5) | ✅ Profil crédible |

Les cinq jeunes défenseurs retenus (Juma Bah, Emmanuel Kakou, Colby Donovan, Andreas Verstraeten, Aurèle Amenda) jouent tous dans leur club enregistré, valent au moins 1 M€ et évoluent dans des clubs vendeurs : le profil le plus rentable selon les Modules 2 et 3.

In [17]:
def comite_recrutement(besoin):
    print("🕵️ Le scout cherche des profils...")
    shortlist = agent("Scout", ROLE_SCOUT, besoin,
                      [chercher_sous_evalues, fiche_joueur, executer_sql])

    print("💰 L'analyste financier évalue les profils...")
    analyse = agent("Analyste", ROLE_ANALYSTE,
                    f"BESOIN DU CLUB :\n{besoin}\n\nPROPOSITIONS DU SCOUT :\n{shortlist}",
                    [fiche_joueur, simuler_amortissement, executer_sql])

    print("✍️ Le rédacteur rédige la note...")
    note = agent("Rédacteur", ROLE_REDACTEUR,
                 f"BESOIN DU CLUB :\n{besoin}\n\nSHORTLIST DU SCOUT :\n{shortlist}"
                 f"\n\nANALYSE FINANCIÈRE :\n{analyse}", [])

    with open("note_recrutement.md", "w", encoding="utf-8") as f:
        f.write(note)
    print("📄 Note enregistrée : note_recrutement.md\n")
    return note

note = comite_recrutement(
    "Le club cherche un défenseur de 23 ans maximum, titulaire dans un championnat "
    "européen, pour un budget maximum de 15 M€. Priorité au potentiel de revente."
)
display(Markdown(note))

🕵️ Le scout cherche des profils...
   ⚠️ gemini-3.5-flash indisponible, passage au modèle de secours...
   🔧 Scout appelle : chercher_sous_evalues
   🔧 Scout appelle : fiche_joueur
   🔧 Scout appelle : fiche_joueur
   🔧 Scout appelle : fiche_joueur
   🔧 Scout appelle : executer_sql
   🔧 Scout appelle : executer_sql
   🔧 Scout appelle : executer_sql
💰 L'analyste financier évalue les profils...
   ⚠️ gemini-3.5-flash indisponible, passage au modèle de secours...
   🔧 Analyste appelle : fiche_joueur
   🔧 Analyste appelle : fiche_joueur
   🔧 Analyste appelle : fiche_joueur
   🔧 Analyste appelle : simuler_amortissement
   🔧 Analyste appelle : simuler_amortissement
   🔧 Analyste appelle : simuler_amortissement
✍️ Le rédacteur rédige la note...
   ⚠️ gemini-3.5-flash indisponible, passage au modèle de secours...
📄 Note enregistrée : note_recrutement.md



# NOTE DE RECRUTEMENT

**Date :** 27/10/2026  
**Émetteur :** Comité de recrutement et Direction financière  

## Résumé exécutif
La recherche cible un défenseur central de 23 ans maximum, titulaire dans un championnat européen, pour un budget $\le 15\text{ M\texteuro}$ avec un fort potentiel de revente. La recommandation finale se porte sur **Juma Bah**, qui présente le meilleur ratio du modèle (2,48) et la plus-value latente la plus élevée (+16,8 M€).

---

## Tableau comparatif des joueurs

| Joueur | Club & Championnat | Âge | Prix d'achat (Val. réelle) | Temps de jeu | Contrat restant | Valeur estimée (Modèle) | Ratio Modèle / Prix |
| :--- | :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **Juma Bah** | OGC Nice (FR1 - France) | 20,2 ans | 10,0 M€ | 2 660 minutes | 5,0 ans | 24,8 M€ | 2,48 |
| **Emmanuel Kakou** | Cercle Brugge (BE1 - Belgique) | 21,0 ans | 2,0 M€ | 1 733 minutes | 1,0 an | 4,5 M€ | 2,24 |
| **Aurèle Amenda** | Eintracht Frankfurt (L1 - Allemagne) | 22,9 ans | 10,0 M€ | 1 899 minutes | 3,0 ans | 17,9 M€ | 1,79 |

---

## Analyse par joueur

### 1. Juma Bah (OGC Nice)
* **Atouts :** Marge de progression optimale (20,2 ans) avant le pic de valorisation. Solide volume de jeu en Ligue 1 (2 660 minutes). Plus-value latente maximale en fin d'année 1 (+16,8 M€) avec une valeur estimée à 24,8 M€ pour un prix d'achat de 10,0 M€ (ratio de 2,48). Contrat longue durée de 5,0 ans.
* **Risques :** Mouvements historiques de transferts complexes par le passé (impliquant notamment Manchester City), nécessitant une vérification juridique de la structure de propriété.
* **Impact financier :** Prix d'achat de 10,0 M€, charge d'amortissement annuelle de 2,00 M€ sur 5 ans (valeur nette comptable de 8,0 M€). Meilleure allocation de capital.

### 2. Emmanuel Kakou (Cercle Brugge)
* **Atouts :** Profil low-cost (prix d'achat de 2,0 M€, valeur estimée à 4,5 M€, ratio de 2,24). Apport offensif intéressant avec 2 buts et 1 passe décisive en 1 733 minutes. 
* **Risques :** 1,0 an de contrat restant (comporte un risque de concurrence accrue à l'achat). Potentiel de revente en valeur absolue plus modeste (4,5 M€).
* **Impact financier :** Investissement minimal avec une charge d'amortissement annuelle de 0,40 M€ sur 5 ans (valeur nette comptable de 1,6 M€) et une plus-value latente de +2,9 M€ en fin d'année 1.

### 3. Aurèle Amenda (Eintracht Frankfurt)
* **Atouts :** Sécurité d'un championnat majeur (Bundesliga) avec 1 899 minutes jouées. 3,0 ans de contrat restants. Prix d'achat de 10,0 M€ pour une valeur estimée à 17,9 M€ (ratio de 1,79).
* **Risques :** Le plus âgé des trois (22,9 ans), approchant de la limite des 23 ans, ce qui réduit sa fenêtre de progression exponentielle et son potentiel de revente comparé aux autres profils.
* **Impact financier :** Prix d'achat de 10,0 M€, charge d'amortissement annuelle de 2,00 M€ sur 5 ans (valeur nette comptable de 8,0 M€) et plus-value latente de +9,9 M€ en fin d'année 1.

---

## Risques et limites de l'analyse
* **Modèle prédictif :** Les estimations reposent sur la performance, l'âge et le temps de jeu, mais ne garantissent ni l'absence future de blessures, ni l'adaptation tactique à un nouveau championnat, ni les performances futures.
* **Facteurs externes :** Les calculs d'amortissement sont linéaires (norme IAS 38 sur un contrat de 5 ans) et ne prennent pas en compte la volatilité des marchés, l'impact des taxes locales sur les plus-values de cession ni les commissions d'agents.
* **Situation juridique (Juma Bah) :** Présence de mouvements croisés complexes dans son historique de transferts (impliquant Manchester City par le passé), bien qu'il soit actuellement enregistré et actif à l'OGC Nice (aucun des trois joueurs n'est actuellement répertorié comme joueur prêté).

---

## Prochaines étapes
1. **Analyse vidéo :** Approfondissement de l'évaluation sportive des profils ciblés.
2. **Visite médicale :** Planification des examens d'aptitude physique.
3. **Négociation :** Lancement des discussions contractuelles et des transferts (en particulier sur la vérification juridique de la structure de propriété pour Juma Bah et la gestion du contrat court pour Emmanuel Kakou).

### Lecture : le comité après correction
**Recommandation : Juma Bah** (OGC Nice, 20,2 ans, valeur 10 M€, estimation 24,8 M€, ratio 2,48, contrat de 5 ans).

| Avant correction | Après correction |
|---|---|
| Jayden Meghoma, joueur prêté, estimation biaisée | Juma Bah, titulaire dans son club, profil cohérent avec les Modules 2 à 4 |

**Vérification des chiffres (Juma Bah) :** amortissement annuel 10 ÷ 5 = 2 M€, valeur nette comptable en fin d'année 1 = 8 M€, plus-value latente = 24,8 − 8 = **+16,8 M€** ✅. Tous les chiffres de la note proviennent des outils Python.

**Points forts :**
- Classement cohérent avec l'analyse par âge (Module 2) : le plus jeune joueur offre la plus forte plus-value latente.
- Diligence : un passage par Manchester City est repéré dans l'historique de Juma Bah, avec une recommandation de vérification juridique (clause de revente éventuelle).

**Points à améliorer :**
- Affichage d'une formule en LaTeX dans le résumé → consigne ajoutée au rédacteur.
- Lecture différente du contrat court d'Emmanuel Kakou (1 an) : levier de négociation pour l'analyste, risque de concurrence pour le rédacteur. Une relecture humaine reste indispensable.

---
# ✅ Conclusion du Module 6 — Agents IA

### Ce qui a été construit
1. **Trois outils Python testés** : requêtes SQL en lecture seule, recherche de joueurs sous-évalués (modèle du Module 4), fiche joueur avec historique de valeur.
2. **Un agent analyste** capable de planifier seul ses étapes (jusqu'à 9 appels d'outils pour une question) et de refuser d'inventer des chiffres.
3. **Un comité de recrutement multi-agents** (scout, analyste financier, rédacteur) qui produit une note de recrutement, avec une simulation d'amortissement IAS 38.
4. **Un système résilient** : limite de temps sur chaque requête et bascule automatique vers un modèle de secours.

### Les leçons clés
- **L'IA ne calcule pas, elle interprète** : tous les chiffres viennent d'outils vérifiables.
- **Les agents amplifient les erreurs des données** : un biais connu (les prêts) a produit une recommandation fausse, mais très convaincante. La solution a été de **corriger les données à la source**.
- **Un ratio est trompeur sur les petites valeurs** : un seuil minimal de 1 M€ a été ajouté.
- **Le modèle ne connaît pas la date du jour** : elle doit lui être fournie.
- **La relecture humaine reste indispensable** : les agents peuvent interpréter différemment une même information.

### Résultat
Le comité recommande **Juma Bah** (OGC Nice), un défenseur de 20 ans estimé à 24,8 M€ pour une valeur de 10 M€, avec une plus-value latente de +16,8 M€ dès la première année.